In [1]:
import json 
import pandas as pd 
from glob import glob 

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
]

files = glob("/home/work/yuna/HPA/data/models/*.jsonl")  # glob("/home/work/yuna/HPA/results/swift-results/*/*.jsonl")+ 
len(files)
conditions =['sys_inst_blind', 'inst_blind', 'blind', ''] 
indicators = ['pid', 'id', 'image_id', 'index']  
check_dir='/home/work/yuna/HPA/results/check' 

dfs = []
for f in files:
    try: 
        df = pd.read_json(f, lines=True)
    except Exception as e : 
        print(e, f)
        continue

    f = f.replace("_vlm", '').replace("_llm", '')  
    df['filename'] = f
    
    hashable_cols = []
    for col in df.columns:
        try:
            # Try to hash the first non-null value
            sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if sample is not None:
                hash(sample)
            hashable_cols.append(col)
        except (TypeError, KeyError):
            pass

    # Drop duplicates based on hashable columns
    if hashable_cols:
        df = df.drop_duplicates(subset=hashable_cols)
    else:
        # Fallback: convert entire rows to JSON strings
        df['_temp_hash'] = df.apply(lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str), axis=1)
        df = df.drop_duplicates(subset=['_temp_hash']).drop(columns=['_temp_hash'])
        
    # df.to_json(f, orient='records', lines=True)
    # model = f.split('/')[-1][:-6]
    model_name = None 
    for model in model_names : 
        model_name = model.split('/')[-1] 
        if model_name in f :   
            df['model'] = model_name  
            f = f.replace(f'{model_name}_', '') 
            break 
    if model_name is None : 
        print(f, 'model is not found')

    condition= '' 
    for c in conditions : 
        if c in f : 
            f = f.replace(f'_{c}', '')  
            condition = c 
            break 
    df['dataset'] = f.split('/')[-1][:-6] # .split('_')[1].split('_')[0]
    df['condition'] = condition

    if "response" not in df.columns: 
        df.rename(columns={'output':'response'}, inplace=True)
    # else: 
    dfs.append(df) 


In [2]:
df = pd.concat(dfs)
# df = df[(df['condition'] == '' )| (df['condition'] == 'inst_blind')]

summary = df.groupby(['dataset', 'condition', 'model', 'filename']).count()['response'].reset_index().sort_values(by=['response', 'model'])
summary = summary.pivot_table(index=['model'], columns=['condition' , 'dataset', ], values=['response'])
summary.to_csv('./inference_progress.csv') # , 'dataset', 'condition'
summary

response                                            \
condition                                                    blind            
dataset                    mmstar spubench   vqa1k   vqa5k  mmstar spubench   
model                                                                         
InternVL3_5-1B             1500.0   2400.0  1000.0  5000.0  1500.0   2400.0   
InternVL3_5-2B             1500.0   2400.0  1000.0  5000.0  1500.0   2400.0   
InternVL3_5-4B               50.0   2400.0  1000.0  5000.0     NaN      NaN   
InternVL3_5-8B             1074.0   2400.0  1000.0  5000.0  1500.0   2400.0   
Qwen3-VL-2B-Instruct       1500.0    281.0  1000.0  5000.0  1500.0   2400.0   
Qwen3-VL-4B-Instruct       1500.0   2400.0  1000.0  5000.0  1440.0      NaN   
Qwen3-VL-8B-Instruct       1500.0   2400.0  1000.0  5000.0     NaN      NaN   
llava-1.5-7b-hf               NaN   2400.0     NaN     NaN     NaN      NaN   
llava-v1.6-mistral-7b-hf   1500.0   2400.0  1000.0  5000.0  1500.0   2400.0   

                                                                              \
condition                                inst_blind                            
dataset                   vqa_1k  vqa_5k     mmstar spubench  vqa_1k  vqa_5k   
model                                                                          
InternVL3_5-1B            1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
InternVL3_5-2B            1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
InternVL3_5-4B            1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
InternVL3_5-8B            1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
Qwen3-VL-2B-Instruct      1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
Qwen3-VL-4B-Instruct      1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   
Qwen3-VL-8B-Instruct         NaN     NaN     1500.0   2400.0  1000.0  5000.0   
llava-1.5-7b-hf              NaN     NaN        NaN   2400.0     NaN     NaN   
llava-v1.6-mistral-7b-hf  1000.0  5000.0     1500.0   2400.0  1000.0  5000.0   

                                         
condition                sys_inst_blind  
dataset                          vqa_5k  
model                                    
InternVL3_5-1B                   5000.0  
InternVL3_5-2B                   5000.0  
InternVL3_5-4B                      NaN  
InternVL3_5-8B                      NaN  
Qwen3-VL-2B-Instruct                NaN  
Qwen3-VL-4B-Instruct                NaN  
Qwen3-VL-8B-Instruct                NaN  
llava-1.5-7b-hf                     NaN  
llava-v1.6-mistral-7b-hf            NaN

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl